In [2]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
import random
import time
import scimap as sm
import anndata as ad
from functools import partial
from helperFunctions import *
from smallestEnclosingCircle import make_circle
from sklearn.mixture import GaussianMixture
from ecd_helperFunctions import *

Running SCIMAP  2.1.1


### Topographical Correlation Map Feature Based Cox-PH Model

**Step 1: Data Preprocessing**
- Load data into correct format for TCM
- Divide data into ROIs and save as .hd5 files<br>

**Step 2: Generate Complete Spatial Randomness Datasets**
- Generate datasets with equal number of cells as each ROI
- Create "paired" CSR datasets for each ROI<br>

**Step 3: Calculate Topographical Correlation Maps**
- Calculate TCM for both real and CSR datasets<br>

**Step 4: Compare Positive and Negative TCM Distributions**
- For positive TCM values, use K-S test to compare real and CSR datasets
- For negative TCM values, use K-S test to compare real and CSR datasets<br>

**Step 5: Rank ROIs based on K-S Score**
- Use rank to select the top 10 ROIs with the most positive correlation and top 10 ROIs with the most negative correlation between cell markers<br>

**Step 6: Calculate the distance between the centroids of the top 10 ROIs**
- Calculate the pairwise distance between the of the top 10 postive and negative ROIs
- This will result pos_TCM_dist_12, pos_TCM_dist_23, pos_TCM_dist_34, etc. for positive TCM values and neg_TCM_dist_12, neg_TCM_dist_23, neg_TCM_dist_34, etc. for negative TCM values<br>

**Step 7: Calculate the Cox-PH model with L2 regularization**
- The model will have 29 covariates added per marker pair that is assessed in the TCM

### **STEP ONE**: Data Preprocessing

In [2]:
# Get all .csv files in one list
data_dir = '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/'
data_parent_dir = os.listdir(data_dir)

file_paths = []
for i in range(len(data_parent_dir)):
    files = os.listdir(data_dir + data_parent_dir[i])
    csv_files = [file for file in files if file.endswith('.csv')]
    for file in csv_files:
        file_path = os.path.join(data_dir, data_parent_dir[i], file)
        file_paths.append(file_path)

print(file_paths)

['/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC07/P37_S35-CRC07.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC35/P37_S78-CRC35.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC14/P37_S46-CRC14.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC26/P37_S63-CRC26.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC21/P37_S58-CRC21.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC19/P37_S51-CRC19.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC13/P37_S45-CRC13.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC32/P37_S75-CRC32.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC38/P37_S81-CRC38.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC33_02/P37_S76_02-CRC33_02.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC04/P37_S32-C

In [ ]:
# Define markers of interest (reduce the file size needed to save)
markers = ['CD45', 'CD4', 'SMA', 'PD-L1', 'Pan-CK']
grid_directory = '/michorlab/ecdyer/multiplex_spatial/crc_grid_data/cph_grids/'

##### Only run the below step if you want to generate NEW grid files for the TCM

In [ ]:
# Apply GMM to each file and save the grid
for f in file_paths:
    df = apply_gmm([f], markers)
    grid_file_name = grid_directory + f.split('/')[-1].split('.')[0] + '_gmm.h5'
    create_tile_rois(df, tile_size=5000, save=True, save_hdf5=grid_file_name)

### **STEP TWO/THREE/FOUR**: Generate Complete Spatial Randomness Datasets, Calculate TCMs, and Calculate K-S Scores
This is very computationally intensive. Do not run this in a notebook on an entire dataset. Instead, run this in a script on a cluster with multithreading. Multithreading functionality can be achieved with the helper function `ecd_helperFunctions.multithread_compare_tcm()`.

In [ ]:
cwd = '/michorlab/ecdyer/multiplex_spatial/crc_grid_data/cph_grids'
short_grid_files = os.listdir(cwd)
grid_files = [os.path.join(cwd, f) for f in short_grid_files]
markers = ['CD45_status', 'PD-L1_status']
keep_cols = ['X_centroid', 'Y_centroid', 'CellID']
labels  = {1: 'Lymphocytes',
            2: 'PD-L1'}
rename_cols_dict = {'X_centroid': 'x', 
                    'Y_centroid': 'y'}
save_tcm_plot_path = '/michorlab/ecdyer/multiplex_spatial/figures/tcm_plots/'
save_csr_plot_path = '/michorlab/ecdyer/multiplex_spatial/figures/csr_tcm_plots/'
save_ks_results_path = '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_pdl1/'
save_tcm_path = '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_pdl1/'
save_csr_tcm_path = '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_pdl1/'

for grid_file in grid_files:
    compare_tcm(grid_file, 
        markers,
        keep_cols,
        labels,
        visualiseStages=False,
        #save_tcm_plot_path=save_tcm_plot_path,
        #save_csr_plot_path=save_csr_plot_path,
        #save_tcm_path=save_tcm_path,
        #save_csr_tcm_path=save_csr_tcm_path,
        save_ks_results_path=save_ks_results_path,
        rename_cols_dict=rename_cols_dict,
        plot_point_cloud=False)

### **STEP FIVE/SIX**: Rank ROIs and Calculate Pairwise Distances

In [17]:
ks_results_dir = '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_sma/'
grid_files = '/michorlab/ecdyer/multiplex_spatial/crc_grid_data/cph_grids/'
save_dir = '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_sma_ranked/'

test_ks_rank_grid_dict = rank_and_calculate_distance(ks_results_dir, 
                                                     grid_files, 
                                                     save_dir=save_dir,
                                                     log_scale_ks=True)

In [ ]:
ks_results_directories = ['/michorlab/ecdyer/multiplex_spatial/tcm_results/pdl1_sma/',
                          '/michorlab/ecdyer/multiplex_spatial/tcm_results/pdl1_cd45/',
                          'michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_sma/']

save_dirs = ['/michorlab/ecdyer/multiplex_spatial/tcm_results/pdl1_sma_ranked',
                '/michorlab/ecdyer/multiplex_spatial/tcm_results/pdl1_cd45_ranked',
                '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_sma_ranked']

grid_files = '/michorlab/ecdyer/multiplex_spatial/crc_grid_data/cph_grids/'

for i in range(len(ks_results_directories)):
    test_ks_rank_grid_dict = rank_and_calculate_distance(ks_results_directories[i], 
                                                         grid_files, 
                                                         save_dir=save_dirs[i],
                                                         log_scale_ks=True)

### STEP SEVEN: Calculate Survival Models
1. Cox-PH model with L2 regularization
2. Reglularized Cox Model with Elastic Net Regularization
3. Random Survival Forest Model

In [35]:
def combine_clinical_and_tcm(clinical_data_file,
                             results_directory,
                             save_path,
                             markers=None,
                             ks_stat='log_ks_stat'):
    clinical_data = pd.read_csv(clinical_data_file)
    # Ensure the columns exist in the DataFrame
    for i in range(1, 21):
        col_name = f'ks_grid_{i}_{markers}'
        if col_name not in clinical_data.columns:
            clinical_data[col_name] = None

    for i in range(1, 20):
        col_name = f'distance_{i}_{markers}'
        if col_name not in clinical_data.columns:
            clinical_data[col_name] = None
    


    for file in os.listdir(results_directory):
        if file.endswith('.csv'):
            tcm_data = pd.read_csv(os.path.join(results_directory, file))
            tcm_data = tcm_data[['distance_between_grids', 'log_ks_stat']]
            tcm_data['Specimen_ID'] = file[2:5]
            # Extract the log_ks_stat column from tcm_data
            tcm_ks_stat = tcm_data[ks_stat]
            distance = tcm_data['distance_between_grids']

            tcm_ks_stat = tcm_ks_stat.astype(str)
            distance = distance.astype(str)
            # Split the log_ks_stat values into individual columns
            ks_grid_values = tcm_ks_stat.str.split(',', expand=True)
            distance_values = distance.str.split(',', expand=True)

            # Find the row in clinical_data with the matching Specimen_ID
            specimen_ID = file[2:5]
            clinical_row_index = clinical_data[clinical_data['Specimen_ID'] == specimen_ID].index
            # Verify that clinical_row_index is not empty and is a valid index
            if clinical_row_index.empty:
                raise ValueError(f"Specimen_ID {specimen_ID} not found in clinical data")
            
            if not clinical_row_index.empty:
                clinical_row_index = clinical_row_index[0]
                # Fill in the ks_grid_1 to ks_grid_20 values in clinical_data
                for i in range(1, 21):
                    clinical_data.at[clinical_row_index, f'ks_grid_{i}_{markers}'] = ks_grid_values[i-1].astype(float)
                for i in range(1, 20):
                    clinical_data.at[clinical_row_index, f'distance_{i}_{markers}'] = distance_values[i-1].astype(float)
    clinical_data.to_csv(save_path, index=False)
    return


In [41]:
def combine_clinical_and_tcm(clinical_data_file,
                             results_directory,
                             save_path,
                             markers=None,
                             ks_stat='log_ks_stat'):
    clinical_data = pd.read_csv(clinical_data_file)
    
    # Ensure the columns exist in the DataFrame
    for i in range(1, 21):
        col_name = f'ks_grid_{i}_{markers}'
        if col_name not in clinical_data.columns:
            clinical_data[col_name] = None

    for i in range(1, 20):
        col_name = f'distance_{i}_{markers}'
        if col_name not in clinical_data.columns:
            clinical_data[col_name] = None

    for file in os.listdir(results_directory):
        if file.endswith('.csv'):
            tcm_data = pd.read_csv(os.path.join(results_directory, file))
            tcm_data = tcm_data[['distance_between_grids', 'log_ks_stat']]
            tcm_data['Specimen_ID'] = file[2:5]
            # Extract the log_ks_stat column from tcm_data
            tcm_ks_stat = tcm_data[ks_stat]
            distance = tcm_data['distance_between_grids']

            tcm_ks_stat = tcm_ks_stat.astype(str)
            distance = distance.astype(str)
            # Split the log_ks_stat values into individual columns
            ks_grid_values = tcm_ks_stat.str.split(',', expand=True)
            distance_values = distance.str.split(',', expand=True)

            # Find the row in clinical_data with the matching Specimen_ID
            specimen_ID = file[2:5]
            print(specimen_ID)
            clinical_data['Specimen_ID'] = clinical_data['Specimen_ID'].astype(str)
            clinical_row_index = clinical_data[clinical_data['Specimen_ID'] == specimen_ID].index
            # Verify that clinical_row_index is not empty and is a valid index
            if clinical_row_index.empty:
                raise ValueError(f"Specimen_ID {specimen_ID} not found in clinical data")
            
            clinical_row_index = clinical_row_index[0]
            # Fill in the ks_grid_1 to ks_grid_20 values in clinical_data
            for i in range(1, 21):
                if i-1 < ks_grid_values.shape[1]:
                    clinical_data.at[clinical_row_index, f'ks_grid_{i}_{markers}'] = float(ks_grid_values[i-1][0])
            for i in range(1, 20):
                if i-1 < distance_values.shape[1]:
                    clinical_data.at[clinical_row_index, f'distance_{i}_{markers}'] = float(distance_values[i-1][0])
    clinical_data.to_csv(save_path, index=False)
    return


In [49]:
def combine_clinical_and_tcm(clinical_data_file,
                             results_directory,
                             save_path,
                             markers=None,
                             ks_stat='log_ks_stat'):
    
    clinical_data = pd.read_csv(clinical_data_file)
    
    # Ensure the columns exist in the DataFrame
    for i in range(1, 21):
        col_name = f'ks_grid_{i}_{markers}'
        if col_name not in clinical_data.columns:
            clinical_data[col_name] = None

    for i in range(1, 20):
        col_name = f'distance_{i}_{markers}'
        if col_name not in clinical_data.columns:
            clinical_data[col_name] = None

    for file in os.listdir(results_directory):
        if file.endswith('.csv'):
            tcm_data = pd.read_csv(os.path.join(results_directory, file))
            tcm_data = tcm_data[['distance_between_grids', 'log_ks_stat']]
            tcm_data['Specimen_ID'] = file[2:5]

            # Extract the log_ks_stat column from tcm_data
            tcm_ks_stat = tcm_data[ks_stat].astype(str).str.split(',')
            distance = tcm_data['distance_between_grids'].astype(str).str.split(',')

            # Convert split series to lists
            tcm_ks_stat_list = tcm_ks_stat.tolist()
            distance_list = distance.tolist()

            # Find the row in clinical_data with the matching Specimen_ID
            specimen_ID = file[2:5]
            clinical_row_index = clinical_data[clinical_data['Specimen_ID'] == specimen_ID].index
            # Verify that clinical_row_index is not empty and is a valid index
            if clinical_row_index.empty:
                raise ValueError(f"Specimen_ID {specimen_ID} not found in clinical data")
            
            clinical_row_index = clinical_row_index[0]

            # Fill in the ks_grid_1 to ks_grid_20 values in clinical_data
            for i in range(1, 21):
                if i-1 < len(tcm_ks_stat_list):
                    value = tcm_ks_stat_list[i-1][0] if len(tcm_ks_stat_list[i-1]) > 0 else None
                    clinical_data.at[clinical_row_index, f'ks_grid_{i}_{markers}'] = float(value) if value else None
            for i in range(1, 20):
                if i-1 < len(distance_list):
                    value = distance_list[i-1][0] if len(distance_list[i-1]) > 0 else None
                    clinical_data.at[clinical_row_index, f'distance_{i}_{markers}'] = float(value) if value else None

    clinical_data.to_csv(save_path, index=False)
    return


In [50]:
clinical_data_file = '/michorlab/ecdyer/multiplex_spatial/orion_crc_clinical_variables.csv'
results_directory = '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_sma_ranked'
markers = 'cd45_sma'
save_path = '/michorlab/ecdyer/multiplex_spatial/tcm_results/tcm_clin_vars.csv'

combine_clinical_and_tcm(clinical_data_file,
                            results_directory,
                            save_path,
                            markers)

Processing file: CRC21_top_ks_results.csv
Initial tcm_data:
   distance_between_grids  log_ks_stat Specimen_ID
0                0.000000     0.704965         C21
1            15088.409808     0.728592         C21
2            18822.219270     0.743801         C21
3            26696.531691     0.749687         C21
4            26148.086225     0.766293         C21
tcm_ks_stat: 0    [0.7049653950693144]
1     [0.728591853411641]
2    [0.7438009728352141]
3    [0.7496869683817965]
4    [0.7662931799331194]
Name: log_ks_stat, dtype: object
distance: 0                   [0.0]
1     [15088.40980808378]
2    [18822.219270429363]
3    [26696.531690753804]
4    [26148.086224552822]
Name: distance_between_grids, dtype: object
tcm_ks_stat_list: [['0.7049653950693144'], ['0.728591853411641'], ['0.7438009728352141'], ['0.7496869683817965'], ['0.7662931799331194'], ['0.8200138223825025'], ['0.8231594789665022'], ['0.823752726291108'], ['0.8287421500935631'], ['0.8513886512176837'], ['0.8577724872683

In [13]:
# Example Code
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sklearn.model_selection import GridSearchCV
from sksurv.metrics import concordance_index_censored

# Prepare the data in the required format
# y should be a structured array with fields 'event' and 'time'
y = np.array([(row['event'], row['time']) for _, row in data.iterrows()], dtype=[('event', bool), ('time', float)])

# Standardize the covariates
X = scaler.fit_transform(data[covariate_columns])

# Define the model with L2 regularization
coxph_model = make_pipeline(
    StandardScaler(),
    CoxPHSurvivalAnalysis(alpha=1.0)  # alpha is the regularization strength
)

# Hyperparameter tuning
param_grid = {'coxphsurvivalanalysis__alpha': np.logspace(-4, 4, 10)}
grid_search = GridSearchCV(coxph_model, param_grid, cv=5)
grid_search.fit(X, y)

# Best model
best_model = grid_search.best_estimator_
best_alpha = grid_search.best_params_['coxphsurvivalanalysis__alpha']
print(f"Best alpha: {best_alpha}")

# Evaluate the model
c_index = concordance_index_censored(y["event"], y["time"], best_model.predict(X))
print(f"Concordance index: {c_index[0]}")